# Diabetes Severity — Multinomial Logistic Regression

## Project 6

A complete three-class classification project using Multinomial Logistic Regression.

**Classes:** 0 = Low, 1 = Moderate, 2 = High

> **Dataset note:** The included CSV is a synthetic educational practice dataset. It is not a clinical dataset and must not be used for diagnosis or treatment.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss, confusion_matrix, classification_report, roc_auc_score, roc_curve

BASE=Path.cwd()
if not (BASE/"dataset").exists(): BASE=BASE.parent
DATA=BASE/"dataset"/"diabetes_severity.csv"; OUT=BASE/"outputs"; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA); display(df.head()); print("Shape:",df.shape)

## 1. Data Quality and EDA

In [ ]:
display(df.info())
display(df.describe().T)
display(pd.DataFrame({"Missing_Count":df.isna().sum(),"Missing_Percent":df.isna().mean()*100}))

In [ ]:
plt.figure(figsize=(7,5)); df["Severity"].value_counts().sort_index().plot(kind="bar")
plt.title("Diabetes Severity Class Distribution"); plt.xlabel("Severity"); plt.ylabel("Patients")
plt.xticks([0,1,2],["Low","Moderate","High"],rotation=0); plt.show()

In [ ]:
for col in ["Glucose","HbA1c"]:
    plt.figure(figsize=(8,5))
    for c in sorted(df.Severity.unique()):
        plt.hist(df.loc[df.Severity==c,col].dropna(),bins=25,alpha=.5,label=f"Class {c}")
    plt.title(f"{col} Distribution by Severity"); plt.xlabel(col); plt.ylabel("Frequency"); plt.legend(); plt.show()

## 2. Prepare Data and Train/Test Split

In [ ]:
target="Severity"; features=[c for c in df.columns if c!=target]
X=df[features]; y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
print("Train:",X_train.shape,"Test:",X_test.shape)

## 3. Preprocessing Pipeline

**Median imputation → StandardScaler → Multinomial Logistic Regression**

Keeping preprocessing inside the pipeline prevents data leakage during cross-validation.

In [ ]:
preprocessor=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
model=Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",LogisticRegression(solver="lbfgs",max_iter=3000,random_state=42))
])
model.fit(X_train,y_train)
y_pred=model.predict(X_test); y_prob=model.predict_proba(X_test)
print("Classes:",model.named_steps["classifier"].classes_)

## 4. Multiclass Evaluation

In [ ]:
metrics=pd.DataFrame([{
    "Accuracy":accuracy_score(y_test,y_pred),
    "Precision_macro":precision_score(y_test,y_pred,average="macro"),
    "Recall_macro":recall_score(y_test,y_pred,average="macro"),
    "F1_macro":f1_score(y_test,y_pred,average="macro"),
    "F1_weighted":f1_score(y_test,y_pred,average="weighted"),
    "Log_Loss":log_loss(y_test,y_prob),
    "ROC_AUC_ovr_macro":roc_auc_score(y_test,y_prob,multi_class="ovr",average="macro")
}])
display(metrics); print(classification_report(y_test,y_pred))
metrics.to_csv(OUT/"multinomial_logistic_metrics.csv",index=False)

## 5. Confusion Matrix

In [ ]:
classes=sorted(y.unique()); cm=confusion_matrix(y_test,y_pred,labels=classes)
display(pd.DataFrame(cm,index=[f"Actual_{c}" for c in classes],columns=[f"Predicted_{c}" for c in classes]))
plt.figure(figsize=(7,6)); plt.imshow(cm); plt.title("Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(range(3),["Low","Moderate","High"]); plt.yticks(range(3),["Low","Moderate","High"])
for i in range(3):
    for j in range(3): plt.text(j,i,cm[i,j],ha="center",va="center")
plt.colorbar(); plt.show()

## 6. One-vs-Rest ROC Curves

In [ ]:
yb=label_binarize(y_test,classes=classes)
plt.figure(figsize=(8,6))
for i,c in enumerate(classes):
    fpr,tpr,_=roc_curve(yb[:,i],y_prob[:,i])
    auc=roc_auc_score(yb[:,i],y_prob[:,i])
    plt.plot(fpr,tpr,label=f"Class {c} AUC={auc:.3f}")
plt.plot([0,1],[0,1],"--"); plt.title("Multiclass ROC — One-vs-Rest")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend(); plt.show()

## 7. Five-Fold Stratified Cross-Validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
r=cross_validate(model,X,y,cv=cv,scoring=["accuracy","precision_macro","recall_macro","f1_macro"],n_jobs=-1)
cv_summary=pd.DataFrame({
    "Metric":["Accuracy","Precision_macro","Recall_macro","F1_macro"],
    "Mean":[r["test_accuracy"].mean(),r["test_precision_macro"].mean(),r["test_recall_macro"].mean(),r["test_f1_macro"].mean()],
    "Std":[r["test_accuracy"].std(),r["test_precision_macro"].std(),r["test_recall_macro"].std(),r["test_f1_macro"].std()]
})
display(cv_summary); cv_summary.to_csv(OUT/"cross_validation_results.csv",index=False)

## 8. Coefficient Interpretation

In [ ]:
coef=model.named_steps["classifier"].coef_
coef_df=pd.DataFrame(coef,index=model.named_steps["classifier"].classes_,columns=features)
display(coef_df)
coef_long=coef_df.reset_index().melt(id_vars="index",var_name="Feature",value_name="Coefficient").rename(columns={"index":"Class"})
coef_long["Exp_Coefficient"]=np.exp(coef_long["Coefficient"])
coef_long.to_csv(OUT/"multinomial_coefficients.csv",index=False)

## 9. Conclusion

The project demonstrates multiclass Logistic Regression from data preparation through evaluation.

**Pipeline:** Data → EDA → missing-value handling → scaling → multinomial Logistic Regression →
class probabilities → multiclass metrics → confusion matrix → One-vs-Rest ROC-AUC →
cross-validation → coefficient interpretation.

This dataset is synthetic and educational only.